## リアルデータ(静大環境)から学習jsonファイルを作成する

In [2]:
# %%
import os
import json
from pathlib import Path

import torch
import torchaudio

# === パス設定（必要ならここだけ編集） ===
BASE_DIR = Path("/home/tsukagoshitoshihiro/workspace/ICASSP/Semi_supervise/dataset/exist_label")
PROJECT_ROOT = BASE_DIR.parent.parent  # → /home/.../Semi_supervise を想定

OUT_WAV_DIR = BASE_DIR / "divide_wav"
OUT_TXT_DIR = BASE_DIR / "divide_txt"
JSON_PATH = BASE_DIR / "exist_label_divided.json"

OUT_WAV_DIR.mkdir(exist_ok=True, parents=True)
OUT_TXT_DIR.mkdir(exist_ok=True, parents=True)

print("BASE_DIR     :", BASE_DIR)
print("PROJECT_ROOT :", PROJECT_ROOT)
print("OUT_WAV_DIR  :", OUT_WAV_DIR)
print("OUT_TXT_DIR  :", OUT_TXT_DIR)
print("JSON_PATH    :", JSON_PATH)


BASE_DIR     : /home/tsukagoshitoshihiro/workspace/ICASSP/Semi_supervise/dataset/exist_label
PROJECT_ROOT : /home/tsukagoshitoshihiro/workspace/ICASSP/Semi_supervise
OUT_WAV_DIR  : /home/tsukagoshitoshihiro/workspace/ICASSP/Semi_supervise/dataset/exist_label/divide_wav
OUT_TXT_DIR  : /home/tsukagoshitoshihiro/workspace/ICASSP/Semi_supervise/dataset/exist_label/divide_txt
JSON_PATH    : /home/tsukagoshitoshihiro/workspace/ICASSP/Semi_supervise/dataset/exist_label/exist_label_divided.json


In [3]:
# %%
# ==== 時間パラメータ ====
SEG_LEN = 10.0     # 10秒セグメント
OVERLAP = 1.0      # 1秒オーバーラップ
STEP = SEG_LEN - OVERLAP  # 9秒

VALID_START_SEC = 10 * 60   # 600秒 = 10分
VALID_END_SEC   = 30 * 60   # 1800秒 = 30分

TARGET_SR = 16000  # 必要に応じて変更

# ==== ラベル変換 ====
# txt -> イベント種
TXT_LABEL_TO_EVENT = {
    "sw": "swallowing",
    "ch": "chewing",
}

# イベント種 -> 記号（text フィールド用）
EVENT_TO_TEXT_TOKEN = {
    "swallowing": "$",
    "chewing": "#",
    # noise, speech, mask も後で増やしたければここに追加
}


In [5]:
# %%
def load_annotations(txt_path: Path):
    """
    1 ファイル分のアノテーションを読み込む
    return: list of dicts
      [{"start": float, "end": float, "label": "sw" or "ch"}, ...]
    """
    anns = []
    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 3:
                continue
            start, end, lab = parts[0], parts[1], parts[2]
            try:
                start = float(start)
                end = float(end)
            except ValueError:
                continue
            anns.append({
                "start": start,
                "end": end,
                "label": lab,
            })
    return anns
# %%
def generate_segments_for_valid_range(valid_start: float, valid_end: float,
                                      seg_len: float = SEG_LEN, step: float = STEP):
    """
    有効区間 [valid_start, valid_end] 内で 10s / 1s overlap セグメントを作る。
    返り値: list of (seg_index, seg_start_sec, seg_end_sec)
    """
    segments = []
    idx = 0
    t = valid_start
    while t + seg_len <= valid_end + 1e-6:
        segments.append((idx, t, t + seg_len))
        idx += 1
        t += step
    return segments

# 動作確認
segs_test = generate_segments_for_valid_range(VALID_START_SEC, VALID_END_SEC)
print("Num segments for valid range:", len(segs_test))
print("First 3 segments:", segs_test[:3])
print("Last  3 segments:", segs_test[-3:])


Num segments for valid range: 133
First 3 segments: [(0, 600, 610.0), (1, 609.0, 619.0), (2, 618.0, 628.0)]
Last  3 segments: [(130, 1770.0, 1780.0), (131, 1779.0, 1789.0), (132, 1788.0, 1798.0)]


In [6]:
# %%
entries = []  # JSON 用のエントリをここに溜める

wav_files = sorted(BASE_DIR.glob("*.wav"))
print("Found wav files:", [p.name for p in wav_files])

for wav_path in wav_files:
    base_stem = wav_path.stem  # 例: "301"
    txt_path = BASE_DIR / f"{base_stem}.txt"
    
    if not txt_path.exists():
        print(f"[WARN] txt not found for {wav_path.name}, skip.")
        continue
    
    print(f"\n=== Processing {wav_path.name} with {txt_path.name} ===")
    
    # ---- 音声読み込み ----
    wav, sr = torchaudio.load(str(wav_path))  # [ch, T]
    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
        sr = TARGET_SR
    num_samples = wav.shape[1]
    total_len_sec = num_samples / sr
    print(f"  Duration: {total_len_sec:.2f} sec, sr = {sr}")
    
    # ---- アノテーション読み込み ----
    anns = load_annotations(txt_path)
    print(f"  Loaded {len(anns)} annotation lines.")
    
    # ---- 10〜30分の範囲だけセグメント生成 ----
    segments = generate_segments_for_valid_range(VALID_START_SEC, VALID_END_SEC,
                                                 seg_len=SEG_LEN, step=STEP)
    print(f"  Num segments in 10-30 min range: {len(segments)}")
    
    for seg_idx, seg_start_global, seg_end_global in segments:
        # グローバル時間 [sec] → サンプル位置
        sample_start = int(seg_start_global * sr)
        sample_end   = int(seg_end_global * sr)
        
        # 安全対策（万一 total_len_sec < 30分 の場合）
        if sample_start >= num_samples:
            continue
        sample_end = min(sample_end, num_samples)
        
        seg_wav = wav[:, sample_start:sample_end]  # [ch, seg_samples]
        
        # セグメントファイル名
        seg_name = f"{base_stem}_seg{seg_idx:03d}.wav"
        out_wav_path = OUT_WAV_DIR / seg_name
        torchaudio.save(str(out_wav_path), seg_wav, sr)
        
        # ---- このセグメント内のアノテーションを抽出 ----
        seg_ann_lines = []
        timestamps = {
            "chewing": [],
            "swallowing": [],
            "noise": [],
            "speech": [],
            "mask": []
        }
        
        for ann in anns:
            ann_s = ann["start"]
            ann_e = ann["end"]
            lab_str = ann["label"]
            
            # セグメントと全く重ならない場合はスキップ
            if ann_e <= seg_start_global or ann_s >= seg_end_global:
                continue
            
            # セグメントにクリップ
            clipped_s = max(ann_s, seg_start_global)
            clipped_e = min(ann_e, seg_end_global)
            if clipped_e <= clipped_s:
                continue
            
            # セグメント内の相対時間 [sec]
            rel_s = clipped_s - seg_start_global
            rel_e = clipped_e - seg_start_global
            
            # divide_txt 用の行
            seg_ann_lines.append((rel_s, rel_e, lab_str))
            
            # JSON 用 timestamps
            event_name = TXT_LABEL_TO_EVENT.get(lab_str)
            if event_name is not None:
                timestamps[event_name].append([rel_s, rel_e])
        
        # ---- divide_txt ファイルを保存 ----
        seg_txt_name = f"{base_stem}_seg{seg_idx:03d}.txt"
        out_txt_path = OUT_TXT_DIR / seg_txt_name
        
        with open(out_txt_path, "w", encoding="utf-8") as f_txt:
            for rel_s, rel_e, lab_str in seg_ann_lines:
                f_txt.write(f"{rel_s:.6f}\t{rel_e:.6f}\t{lab_str}\n")
        
        # ---- text フィールド用のシンボル列を作成 ----
        # chewing + swallowing をまとめて開始時刻順にソート
        interval_for_text = []
        for ev_name, ivals in timestamps.items():
            if ev_name not in EVENT_TO_TEXT_TOKEN:
                continue
            token = EVENT_TO_TEXT_TOKEN[ev_name]
            for (s, e) in ivals:
                interval_for_text.append((s, e, token))
        
        interval_for_text.sort(key=lambda x: x[0])  # start でソート
        
        text_seq = "".join(tok for _, _, tok in interval_for_text)
        
        # ---- JSON 用 path は PROJECT_ROOT からの相対パスにする ----
        rel_wav_path = out_wav_path.relative_to(PROJECT_ROOT)
        
        entry = {
            "path": str(rel_wav_path),  # 例: "dataset/exist_label/divide_wav/301_seg000.wav"
            "timestamps": timestamps,
            "text": text_seq,
        }
        entries.append(entry)

print("\nTotal JSON entries:", len(entries))


Found wav files: ['301.wav', '307.wav', '309.wav']

=== Processing 301.wav with 301.txt ===
  Duration: 5400.00 sec, sr = 16000
  Loaded 377 annotation lines.
  Num segments in 10-30 min range: 133

=== Processing 307.wav with 307.txt ===
  Duration: 5400.00 sec, sr = 16000
  Loaded 419 annotation lines.
  Num segments in 10-30 min range: 133

=== Processing 309.wav with 309.txt ===
  Duration: 5400.00 sec, sr = 16000
  Loaded 420 annotation lines.
  Num segments in 10-30 min range: 133

Total JSON entries: 399


In [7]:
# %%
# JSON 保存
with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(entries, f, ensure_ascii=False, indent=2)

print("Saved JSON to:", JSON_PATH)


Saved JSON to: /home/tsukagoshitoshihiro/workspace/ICASSP/Semi_supervise/dataset/exist_label/exist_label_divided.json


In [8]:
# %%
# ざっと中身を確認
print("First 3 entries:")
for e in entries[:3]:
    print(json.dumps(e, ensure_ascii=False, indent=2))
    print("-" * 40)


First 3 entries:
{
  "path": "dataset/exist_label/divide_wav/301_seg000.wav",
  "timestamps": {
    "chewing": [],
    "swallowing": [],
    "noise": [],
    "speech": [],
    "mask": []
  },
  "text": ""
}
----------------------------------------
{
  "path": "dataset/exist_label/divide_wav/301_seg001.wav",
  "timestamps": {
    "chewing": [],
    "swallowing": [
      [
        4.1299999999999955,
        4.779999999999973
      ]
    ],
    "noise": [],
    "speech": [],
    "mask": []
  },
  "text": "$"
}
----------------------------------------
{
  "path": "dataset/exist_label/divide_wav/301_seg002.wav",
  "timestamps": {
    "chewing": [],
    "swallowing": [],
    "noise": [],
    "speech": [],
    "mask": []
  },
  "text": ""
}
----------------------------------------


In [11]:
# %%
# 既存の divided JSON から読み直す場合

JSON_PATH = BASE_DIR / "exist_label_divided.json"

with open(JSON_PATH, "r", encoding="utf-8") as f:
    entries = json.load(f)

print("Loaded entries:", len(entries))

import random
from math import floor

random.seed(42)
num_all = len(entries)
num_valid = max(1, floor(num_all * 0.3))

indices = list(range(num_all))
random.shuffle(indices)

valid_indices = set(indices[:num_valid])
train_indices = set(indices[num_valid:])

train_entries = [entries[i] for i in range(num_all) if i in train_indices]
valid_entries = [entries[i] for i in range(num_all) if i in valid_indices]

print("Total:", num_all)
print("Train:", len(train_entries))
print("Valid:", len(valid_entries))

TRAIN_JSON_PATH = BASE_DIR / "exist_label_train.json"
VALID_JSON_PATH = BASE_DIR / "exist_label_valid.json"

with open(TRAIN_JSON_PATH, "w", encoding="utf-8") as f_tr:
    json.dump(train_entries, f_tr, ensure_ascii=False, indent=2)

with open(VALID_JSON_PATH, "w", encoding="utf-8") as f_va:
    json.dump(valid_entries, f_va, ensure_ascii=False, indent=2)

print("Saved train JSON:", TRAIN_JSON_PATH)
print("Saved valid JSON:", VALID_JSON_PATH)


Loaded entries: 399
Total: 399
Train: 280
Valid: 119
Saved train JSON: /home/tsukagoshitoshihiro/workspace/ICASSP/Semi_supervise/dataset/exist_label/exist_label_train.json
Saved valid JSON: /home/tsukagoshitoshihiro/workspace/ICASSP/Semi_supervise/dataset/exist_label/exist_label_valid.json
